In [1]:
from mdcrow import MDCrow
from langchain.callbacks import get_openai_callback
from datetime import datetime
import os 
import traceback

In [2]:
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(parent_dir)
from robustness_prompts import get_prompt # noqa: E402

prompt_7_natural = get_prompt("natural", 7)

prompt_7_natural

'Simulate 1LYZ for 1ps at 300 K. Report the secondary structure assignments of the downloaded PDB structure and any known binding or active sites. Compute the RMSD, SASA, and radius of gyration of the simulation.'

In [3]:
llm_model = "gpt-4o-2024-08-06"
tools = "all"

In [4]:
agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools, 
    use_memory=False,
    streaming=False,
    verbose=True,
)
with get_openai_callback() as cb:
    chat_start = datetime.now()
    try:
        response = agent.run(prompt_7_natural, callbacks=[cb])
        print(response)
    except Exception as e:
        exc_type, exc_value, exc_traceback = sys.exc_info()
        print(f'{type(e).__name__}:{e}')
        print("".join(traceback.format_exception(exc_type, exc_value, exc_traceback)))
    chat_end = datetime.now()
    total_runtime = (chat_end - chat_start).total_seconds()
    print(cb)
    print(f"Total runtime: {total_runtime:.2f}s")



> Entering new AgentExecutor chain...


Thought: To address the question, I need to perform several tasks: download the PDB file for 1LYZ, simulate it for 1 picosecond at 300 K, report the secondary structure assignments, identify any known binding or active sites, and compute the RMSD, SASA, and radius of gyration. I will start by downloading the PDB file for 1LYZ.

Action:
```json
{
  "action": "PDBFileDownloader",
  "action_input": "1LYZ"
}
```PDB file found with this ID: 1LYZ

Observation: Succeeded. Downloaded the PDB file:1LYZ_214604
Thought:

To proceed, I need to set up and run a short simulation of the downloaded PDB file (1LYZ_214604) for 1 picosecond at 300 K. After the simulation, I will analyze the secondary structure, binding sites, active sites, RMSD, SASA, and radius of gyration.

Action:
```json
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "1LYZ_214604",
    "forcefield_files": ["amber99sb.xml", "tip3p.xml"],
    "save": true,
    "system_params": {
      "nonbondedMethod": "NoCutoff",
      "constraints": "HBonds",
      "rigidWater": true,
      "solvate": true
    },
    "integrator_params": {
      "integrator_type": "LangevinMiddle",
      "Temperature": "300 * kelvin",
      "Friction": "1.0 / picoseconds",
      "Timestep": "0.002 * picoseconds"
    },
    "simulation_params": {
      "Ensemble": "NVT",
      "Number of Steps": 500,
      "record_interval_steps": 50,
      "record_params": ["step", "potentialEnergy", "temperature"]
    }
  }
}
```error found: constraintTolerance mu

To correct the issue, I need to specify the `constraintTolerance` parameter since `rigidWater` is set to `True`. I will update the simulation setup with an appropriate `constraintTolerance` value and rerun the simulation setup.

Action:
```json
{
  "action": "SetUpandRunFunction",
  "action_input": {
    "pdb_id": "1LYZ_214604",
    "forcefield_files": ["amber99sb.xml", "tip3p.xml"],
    "save": true,
    "system_params": {
      "nonbondedMethod": "NoCutoff",
      "constraints": "HBonds",
      "rigidWater": true,
      "constraintTolerance": 0.00001,
      "solvate": true
    },
    "integrator_params": {
      "integrator_type": "LangevinMiddle",
      "Temperature": "300 * kelvin",
      "Friction": "1.0 / picoseconds",
      "Timestep": "0.002 * picoseconds"
    },
    "simulation_params": {
      "Ensemble": "NVT",
      "Number of Steps": 500,
      "record_interval_steps": 50,
      "record_params": ["step", "potentialEnergy", "temperature"]
    }
  }
}
```Building system...


About to create system...
Attempts at creating system: 0/3
Error adding solvent ValueError – No template found for residue 1 (LYS).  The set of atoms is similar to HID, but it is missing 8 atoms.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#template
Residue code: LYS
Residue code not in solvent list. Adding forcefield                         not supported.

Observation: Failed. No template found for residue 1 (LYS).  The set of atoms is similar to HID, but it is missing 8 atoms.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#templateThis were the inputs {'pdb_id': '1LYZ_214604', 'forcefield_files': ['amber99sb.xml', 'tip3p.xml'], 'save': True, 'system_params': {'nonbondedMethod': 'NoCutoff', 'constraints': 'HBonds', 'rigidWater': True, 'constraintTolerance': 1e-05, 'solvate': True}, 'integrator_params': {'integrator_type': 'LangevinMiddle', 'Temperature': '300 * kelvin', 'Friction': '1.0 / picos

...

Minimization complete!


Initial Positions saved to initial_positions.pdb
Equilibrating...


Simulating...


Done!
[2025-12-22T23:16:50.788536] Simulation ended
Total simulation time: 5429.79s
Constraints must be set to 'None' for LangevinMiddle integrator.
LangevinMiddle constraints:  HBonds
Standalone simulation script written to /gpfs/fs2/scratch/qcampbe2/repos/MDCrow/ckpt/ckpt_24/simulations/NVT_1LYZ_214613_20251222_214615.py
['copy_prompt6.ipynb', 'copy_prompt8.ipynb', 'copy_prompt7.ipynb', 'prompt1.ipynb', 'results_gpt-4o-2024-08-06', 'prompt3.ipynb', 'copy_prompt2.ipynb', '.ipynb_checkpoints', 'prompt10.ipynb', 'prompt9.ipynb', 'TRAJ_sim0_214615_1LYZ_214613_20251222_214620.dcd', 'copy_prompt5.ipynb', 'prompt2.ipynb', 'copy_prompt3.ipynb', 'prompt8.ipynb', 'copy_prompt4.ipynb', 'copy_prompt10.ipynb', 'prompt6.ipynb', 'TOP_sim0_214615_1LYZ_214613_20251222_214620.pdb', 'prompt4.ipynb', 'LOG_sim0_214615_1LYZ_214613_20251222_214620.txt', 'copy_prompt9.ipynb', 'prompt5.ipynb']

Observation: Succeeded. Simulation done! 
 Summary: 
[('rec0_214620', 'Simulation trajectory for protein 1LYZ_21461

To complete the analysis, I need to perform the following tasks:

1. Compute the secondary structure assignments using the DSSP tool.
2. Identify any known binding or active sites for the protein.
3. Compute the RMSD, SASA, and radius of gyration for the simulation.

I will start by computing the secondary structure assignments using the DSSP tool.

> Finished chain.
({'input': '\n    You are an expert molecular dynamics scientist, and\n    your task is to respond to the question or\n    solve the problem to the best of your ability using\n    the provided tools.\n\n    You can only respond with a single complete\n    \'Thought, Action, Action Input\' format\n    OR a single \'Final Answer\' format.\n\n    Complete format:\n    Thought: (reflect on your progress and decide what to do next)\n    Action:\n    ```\n    {\n        "action": (the action name, it should be the name of a tool),\n        "action_input": (the input string for the action)\n    }\n    \'\'\'\n\n    OR\n\n    Fina

In [3]:
registry = agent.path_registry
print(registry.list_path_names_and_descriptions().replace(",", "\n"))

Files found in registry: 1LYZ_214604: PDB file downloaded from RSCB
 PDBFile ID: 1LYZ_214604
 1LYZ_214613: Cleaned File:  Removed Heterogens
 and Water Kept.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 
 top_sim0_214615: Initial positions for simulation sim0_214615
 sim0_214615: Basic Simulation of Protein 1LYZ_214613
 rec0_214620: Simulation trajectory for protein 1LYZ_214613 and simulation sim0_214615
 rec1_214620: Simulation state log for protein 1LYZ_214613 and simulation sim0_214615
 rec2_214620: Simulation pdb frames for protein 1LYZ_214613 and simulation sim0_214615


In [4]:
# make sure pdb was downloaded
assert os.path.exists(registry.get_mapped_path("1LYZ_214604"))

In [5]:
# # make sure dssp was computed correctly
# from mdcrow.tools.base_tools import ComputeDSSP

# dssp = ComputeDSSP(registry)
# dssp._run(traj_file= "1LYZ_125306", target_frames="first")

In [6]:
# # make sure the sites were found
# from mdcrow.tools.base_tools import GetAllKnownSites

# get_all_known_sites = GetAllKnownSites()
# get_all_known_sites._run(query="1LYZ", primary_accession="")

In [10]:
# make sure trajectory and topology exist
traj_path = registry.get_mapped_path("rec0_214620")
top_path = registry.get_mapped_path("top_sim0_214615")

assert os.path.exists(traj_path)
assert os.path.exists(top_path)

In [7]:
# # make sure rmsd plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_022913'))

In [8]:
# # make sure rgy plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_023033'))

In [9]:
# # make sure sasa plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_023320'))

In [14]:
#verify the total cost
def calculate_llm_cost(input_tokens, output_tokens, model):
    pricing_2024 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 5/1e6, "output": 15/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
        "claude-3-opus": {"input": 15/1e6, "output": 75/1e6},
        "claude-3.5-sonnet": {"input": 3/1e6, "output": 15/1e6},
    }
    pricing_2025 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 2.5/1e6, "output": 10/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
    }
    
    prices = pricing_2025[model]
    cost = (input_tokens * prices["input"]) + (output_tokens * prices["output"])
    return round(cost, 6)

llm_cost = calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name)
print('Input tokens:',cb.prompt_tokens)
print('Output tokens:',cb.completion_tokens)
print('LLM costs: $',llm_cost)

Input tokens: 50043
Output tokens: 1189
LLM costs: $ 0.136998
